In [ ]:
import numpy as np 
import pandas as pd 
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import plotly.express as px
import plotly.graph_objects as go 
import ast
import re

In [128]:
df = pd.read_csv("metadata.csv")
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [165]:
rating = pd.read_csv("new_ratings")

In [15]:
import warnings
warnings.filterwarnings('ignore')

# cleaning metadata

In [129]:
df.drop_duplicates(inplace= True)

In [130]:
df[["volumes","chapters"]] = df[["volumes","chapters"]].fillna(0)

In [131]:
df["stats"] = df["stats"].apply(ast.literal_eval)

### calculating actual avg score

In [137]:
# to fill the average score column NaN values and calculating the actual avg score
def average_score(data): 
    distribution = data.get('scoreDistribution') or []
    total_votes = sum(item['amount'] for item in distribution)
    if total_votes == 0:
        return None 
    total_score = sum(item['score'] * item['amount'] for item in distribution)
    return total_score / total_votes

In [136]:
df.info()

<class 'pandas.DataFrame'>
Index: 155839 entries, 0 to 155949
Data columns (total 31 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               155839 non-null  int64  
 1   idMal            89632 non-null   float64
 2   title            155839 non-null  str    
 3   type             155839 non-null  str    
 4   format           155704 non-null  str    
 5   status           155786 non-null  str    
 6   startDate        155839 non-null  str    
 7   endDate          155839 non-null  str    
 8   season           14333 non-null   str    
 9   seasonYear       14333 non-null   float64
 10  episodes         21391 non-null   float64
 11  chapters         155839 non-null  float64
 12  volumes          155839 non-null  float64
 13  duration         21206 non-null   float64
 14  averageScore     46396 non-null   float64
 15  popularity       155839 non-null  int64  
 16  trending         155839 non-null  int64  
 17  descrip

In [138]:
calculated_averages = df["stats"].apply(average_score)
df["averageScore"] = df["averageScore"].fillna(calculated_averages)

df["averageScore"] = df["averageScore"].fillna(0)

In [139]:
df["averageScore"] = df["averageScore"].fillna(0)

### fill the years null values

In [140]:
# filling the null seasonYear values 
df["startDate"] = df["startDate"].apply(ast.literal_eval)
year = df["startDate"].apply(lambda x: x.get("year"))
df["seasonYear"] = df["seasonYear"].fillna(year)

### droping unneccessary columns
##### media columns will use later

In [141]:
df.drop(columns = ["startDate","endDate", "duration","coverImage","bannerImage","trailer","siteUrl","hashtag","season"], inplace= True)

### converting the string datatypes to objects

In [142]:
# converting datatype into objects
df["title"] = df["title"].apply(ast.literal_eval)
df["synonyms"] = df["synonyms"].apply(ast.literal_eval)
df["studios"] = df["studios"].apply(ast.literal_eval)
df["genres"] = df["genres"].apply(ast.literal_eval)
df["tags"] = df["tags"].apply(ast.literal_eval)

### merging titles and columns to form a list which will contain only eng,jp and same short names

In [143]:
#extracting the values and make it list
def title_listing(df): 
    title = []
    for i in df.values():
        if i == None :
            break
        title.append(i)
    return title

In [144]:
df["title"] = df["title"].apply(title_listing)

In [145]:
# extracting only english synonyms
def extracting_eng(data): 
    return [t for t in data if re.match(r'^[a-zA-Z\s\'\-]+$', t)]

In [146]:
df["synonyms"] = df["synonyms"].apply(extracting_eng)

In [148]:
df["title"] = df["title"] + df["synonyms"]

In [149]:
df.drop(columns = "synonyms", inplace= True)

In [150]:
# converting into lower case
df["title"] = df["title"].apply(lambda x: [item.lower() for item in x] if isinstance(x, list) else x)

### calculating the total users who rates

In [153]:
def total_amount(data):
    if not isinstance(data, dict):
        return 0
    return sum(item.get('amount', 0) for item in (data.get('scoreDistribution') or []))

In [154]:
df["stats"] = df["stats"].apply(total_amount)

### converting sutudio values into the list

In [155]:
# extracting studio names
def studio_names(data):
    d = data["nodes"]
    studio = []
    for i in d: 
        studio.append(i["name"])
    return studio

df["studios"] = df["studios"].apply(studio_names)

### filling some nan values

In [162]:
df["source"] = df["source"].fillna("UNKNOWN")
df["description"] = df["description"].fillna("")

df["seasonYear"] = df["seasonYear"].fillna(0000.0)

df["episodes"] = df["episodes"].fillna(0)

df["format"] = df["format"].fillna("UNKNOWN")

df["status"] = df["status"].fillna("UNKNOWN")
df["idMal"] = df["idMal"].fillna(-1)

df[df["id"] == 21]["episodes"] = df[df["id"] == 21]["episodes"].fillna(1156)

### creating geners using tags 

In [157]:
def tags_filter(df) :
    tags = []
    for item in df : 
        if item["rank"] > 50 :
            tags.append(item["name"])
    return tags

df["tags"] = df["tags"].apply(tags_filter)

df["genres"] = df["tags"] + df["genres"]

df.drop(columns= "tags", inplace= True)

df["genres"] = df["genres"].apply(lambda x: [item.lower() for item in x] if isinstance(x, list) else x)

In [172]:
df["averageScore"] = (df["averageScore"] / 10).round(1)

In [ ]:
df.rename(columns= 
         {
             "averageScore":"rating",
             "stats" : "voteCount",
             "seasonYear" : "year"
             
         }, inplace= True)

df[["idMal","year", "episodes"]] = df[["idMal","year", "episodes"]].astype(int)

### cleaning description

In [33]:
df["description"] = df["description"].str.lower()
df["description"] = df["description"].fillna("").apply(cs.fix)

In [35]:
def text_clean(data): 
    data =  re.sub(r"<br\s*/?>|'s\b|https?://\S+|www\.\S+ ","",data)
    return re.findall(r'[a-zA-Z]+', data)
df["description"] = df["description"].apply(text_clean)

In [41]:
# removing stop words
stopword = stopwords.words("english")
def stop_word (data): 
    new_data = [word for word in data if word not in stopword]
    return new_data
df["description"] = df["description"].apply(stop_word)

In [44]:
# stemming 
ps = PorterStemmer()

def stemming(data): 
    y = [ps.stem(word) for word in data ]
    return y
df["description"] = df["description"].apply(stemming)

In [45]:
# again converting to string
def back_to_string(data):
    return " ".join(data)

df["description"] = df['description'].apply(back_to_string)

In [54]:
df.to_csv("cleaned_metadata.csv",index = False)

In [2]:
df = pd.read_csv("cleaned_metadata.csv")